# Vessel Ops AI — Unsloth Fine-tune: Gemma 4 × WHO IMGS

Fine-tunes `gemma-4-2b` on a Q&A dataset derived from the WHO *International Medical Guide for Ships* (3rd Edition) using Unsloth + QLoRA.

**Kaggle runtime:** T4 GPU (16 GB VRAM) · ~2–3 hours to train

**Output:** LoRA adapter + merged GGUF pushed to HuggingFace Hub for use with Ollama

---
**Tracks targeted:** Unsloth Special Technology Prize · Health & Sciences · Global Resilience

In [ ]:
# Install dependencies (Kaggle GPU runtime)
!pip install unsloth datasets huggingface_hub -q
!pip install --upgrade transformers -q

In [ ]:
import json, os
from pathlib import Path
from datasets import Dataset
from huggingface_hub import login

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID       = "unsloth/gemma-4-E2B-it"          # Unsloth-optimised Gemma 4 2B instruct
LORA_RANK      = 16
MAX_SEQ_LEN    = 2048
BATCH_SIZE     = 2
GRAD_ACCUM     = 4                              # effective batch = 8
EPOCHS         = 3
LR             = 2e-4
OUTPUT_DIR     = "/kaggle/working/vessel-ops-gemma4"
HF_REPO_ID     = "vessel-ops-ai/gemma4-maritime-medical"  # update to your HF org
DATASET_PATH   = "/kaggle/input/vessel-ops-who-imgs/who_imgs_qa.jsonl"  # upload as Kaggle dataset

# HuggingFace token — add as a Kaggle Secret named HF_TOKEN
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
records = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} training examples")
print("Sample:")
sample = records[42]["conversations"]
for turn in sample:
    print(f"  [{turn['from']}]: {turn['value'][:120]}...")

In [ ]:
# ── Load model with Unsloth ───────────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto-detect: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,
)
print(model.print_trainable_parameters())

In [ ]:
# ── Format conversations into Gemma chat template ─────────────────────────────
def format_conversation(example):
    convs = example["conversations"]
    # Build a list of {role, content} dicts
    messages = []
    for turn in convs:
        role_map = {"system": "system", "human": "user", "gpt": "assistant"}
        role = role_map.get(turn["from"], turn["from"])
        messages.append({"role": role, "content": turn["value"]})
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

raw_dataset = Dataset.from_list(records)
dataset = raw_dataset.map(format_conversation, remove_columns=raw_dataset.column_names)
dataset = dataset.filter(lambda x: len(x["text"]) < MAX_SEQ_LEN * 4)  # rough char filter

# 90/10 train/val split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        packing=True,                    # Unsloth packing for efficiency
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete. Peak VRAM: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

In [ ]:
# ── Quick inference test ──────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_questions = [
    "A crew member has severe chest pain radiating to his left arm. He is sweating and pale. What should I do?",
    "How do I treat a deep laceration that won't stop bleeding?",
    "A sailor has symptoms of appendicitis 400 miles from shore. What are the signs and what can I give them?",
]

SYSTEM = (
    "You are an expert maritime medicine assistant trained on the WHO International "
    "Medical Guide for Ships (3rd Edition). You provide accurate, concise medical "
    "guidance to ship officers managing emergencies at sea with no doctor available. "
    "Always cite the relevant WHO IMGS page number when referencing specific protocols."
)

for q in test_questions:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {response}\n{'='*60}")

In [ ]:
# ── Save LoRA adapter to HuggingFace Hub ─────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"LoRA adapter pushed to hf.co/{HF_REPO_ID}")

In [ ]:
# ── Export merged GGUF (Q4_K_M) for Ollama ────────────────────────────────────
# This lets the fine-tuned model run directly in the Vessel Ops AI app via Ollama.
MERGED_DIR  = "/kaggle/working/vessel-ops-gemma4-merged"
GGUF_REPO   = HF_REPO_ID + "-GGUF"

model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf(
    MERGED_DIR,
    tokenizer,
    quantization_method="q4_k_m",   # Good quality/size tradeoff for shipboard hardware
)
model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
    token=HF_TOKEN,
)
print(f"GGUF model pushed to hf.co/{GGUF_REPO}")
print("To use with Ollama: ollama run hf.co/" + GGUF_REPO)

## Integrating the Fine-tuned Model with Vessel Ops AI

Once the GGUF is on HuggingFace, swap it into the app in two ways:

**Option A — Ollama (recommended for desktop companion)**
```bash
ollama run hf.co/vessel-ops-ai/gemma4-maritime-medical-GGUF
# then update MODEL_PRIMARY in .env:
MODEL_PRIMARY=vessel-ops-ai/gemma4-maritime-medical-GGUF
```

**Option B — direct GGUF via llama.cpp**
```bash
# Bundled in installer, no Ollama required (smaller dependency footprint)
llama-server -m vessel-ops-gemma4.Q4_K_M.gguf --port 11434
```

The fine-tuned model internalises the WHO IMGS corpus, so even without the BM25 RAG layer it produces page-cited maritime medical guidance. With RAG enabled on top, hallucination is further reduced.